
# PSD Parameter Estimation Across an Extended 3D Linear PDE Library — Complete Master

This is the **3D counterpart of the cleaned 1D and 2D notebooks**.

For a 3D field \(u(x,y,z,t)\), Fourier modes are indexed by
\(\mathbf{k}=(k_x,k_y,k_z)\), and for isotropic operators we use

\[
k=\|\mathbf{k}\|=\sqrt{k_x^2+k_y^2+k_z^2}.
\]

For a first-order linear PDE,

\[
\widehat u(\mathbf{k},t)
=
\widehat u(\mathbf{k},0)e^{\lambda(\mathbf{k})t},
\qquad
P(\mathbf{k},t)
=
P(\mathbf{k},0)e^{2\operatorname{Re}\lambda(\mathbf{k})t}.
\]

The notebook uses four 3D synthetic initial conditions, the same 53-model linear library as the cleaned 1D notebook, all five inference scenarios, per-PDE five-scenario plots, and final cross-PDE comparison plots.

**3D-specific choices**

- isotropic PSDs are radially averaged over spherical shells;
- advection and third-order dispersion are implemented as \(x\)-directed phase controls;
- Scenario 5 divides a volume into non-overlapping **3D cubes**;
- visualization uses central \(z\)-slices;
- the development grid is \(32^3\) to keep the notebook practical.


## 0. Imports and global controls

In [ ]:

import os, math, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit
from scipy.stats import linregress
from numpy.fft import fftn, ifftn, fftfreq
from IPython.display import display

np.random.seed(42)
plt.rcParams.update({"figure.dpi":110,"font.size":10})

FULL_RESOLUTION=False
n=64 if FULL_RESOLUTION else 32

L_um=2500.0
dx_um=L_um/n

dt=0.05
n_frames=30
times=np.arange(n_frames)*dt

FRAME_LIST=[5,10,15,20,25,29]
FRAME_T=25
SNAP_FRAME=28

K_MAX=0.08
K_MIN_SNAPSHOT=0.005
K_MAX_SNAPSHOT=0.15

R2_MIN=0.80
GRID_N=2
GRID_LIST=[2,3,4]

KAPPA_SH=1.0

RUN_PER_MODEL_FIGURES=False
SAVE_FIGURES=False
FIG_DIR="master_3d_figures"

if SAVE_FIGURES:
    os.makedirs(FIG_DIR,exist_ok=True)

print(f"Grid: {n}^3 = {n**3:,} voxels")
print(f"dx = {dx_um:.4f} um")
print(f"snapshot time = {times[SNAP_FRAME]:.4f}")


## 1. Elementary 3D operators and complete 53-model library

In [ ]:

ELEMENTARY={
    "Diff":{"params":{"D":750.0}},
    "Adv":{"params":{"c":120.0}},
    "Hyper":{"params":{"nu":2.0e5}},
    "SH":{"params":{"ell":22.0}},
    "FracDiff":{"params":{"Df":350.0,"alpha":1.5}},
    "Disp":{"params":{"beta3":8.0e4}},
    "Sixth":{"params":{"mu6":4.0e7}},
    "Loss":{"params":{"eta":0.20}},
    "Nonlocal":{"params":{"chi":0.80,"sigma":45.0}},
}
ELEMENTARY_NAMES=list(ELEMENTARY)

MODEL_LIBRARY={}

for opname in ELEMENTARY_NAMES:
    tag="R+"+opname
    true={"R":1.25 if opname=="SH" else 0.65,**ELEMENTARY[opname]["params"]}
    MODEL_LIBRARY[tag]={"kind":"elementary","ops":(opname,),"true":true}

for op1,op2 in itertools.combinations(ELEMENTARY_NAMES,2):
    tag=f"R+{op1}+{op2}"
    true={"R":1.25 if "SH" in (op1,op2) else 0.65}
    true.update(ELEMENTARY[op1]["params"])
    true.update(ELEMENTARY[op2]["params"])
    MODEL_LIBRARY[tag]={"kind":"pairwise","ops":(op1,op2),"true":true}

CORE=("Diff","Adv","Hyper","SH")
for n_ops in (3,4):
    for ops in itertools.combinations(CORE,n_ops):
        tag="R+"+"+".join(ops)
        true={"R":1.25 if "SH" in ops else 0.65}
        if "Diff" in ops: true["D"]=300.0 if "SH" in ops else 500.0
        if "Adv" in ops: true["c"]=120.0
        if "Hyper" in ops: true["nu"]=1.0e5
        if "SH" in ops: true["ell"]=22.0
        MODEL_LIBRARY[tag]={"kind":"core_mixed","ops":ops,"true":true}

MODEL_LIBRARY["R+KSlin"]={
    "kind":"KS","ops":("KSlin",),
    "true":{"R":0.65,"a2":450.0,"a4":1.5e5}
}
MODEL_LIBRARY["CHlin"]={
    "kind":"CH","ops":("CHlin",),
    "true":{"A2":450.0,"A4":1.5e5}
}
MODEL_LIBRARY["R+Mixed"]={
    "kind":"mixed","ops":("Diff","Adv","Disp","Hyper"),
    "true":{"R":0.65,"D":500.0,"c":120.0,"beta3":8.0e4,"nu":1.0e5}
}

MODEL_VARIANTS=list(MODEL_LIBRARY)
assert len(MODEL_VARIANTS)==53

print("Total models:",len(MODEL_VARIANTS))
display(pd.DataFrame([
    {"PDE":tag,"Kind":spec["kind"],"Operators":" + ".join(spec["ops"]),"True parameters":spec["true"]}
    for tag,spec in MODEL_LIBRARY.items()
]))



## 2. Special second-order / delay models

The telegraph and delay-diffusion equations are retained as separate documented families because they do not share the first-order additive-symbol structure used by Scenarios 1–5.


In [ ]:

SPECIAL_MODELS={
    "R+Telegraph":{
        "equation":"u_tt + tau^{-1}u_t = cT^2 Laplacian(u) + R u",
        "true":{"R":0.50,"tau":0.80,"cT":90.0}
    },
    "R+DelayDiff":{
        "equation":"u_t = R u(x,t-tau) + D Laplacian(u)",
        "true":{"R":0.50,"D":500.0,"tau":0.50}
    }
}
display(pd.DataFrame(SPECIAL_MODELS).T)


## 3. Four periodic 3D synthetic initial conditions

In [ ]:

coords=np.linspace(0,L_um,n,endpoint=False)
X,Y,Z=np.meshgrid(coords,coords,coords,indexing="ij")

def pdist(a,b,L=L_um):
    d=np.abs(a-b)
    return np.minimum(d,L-d)

def gaussian3_periodic(cx,cy,cz,sigma,amplitude=1.0):
    dx=pdist(X,cx); dy=pdist(Y,cy); dz=pdist(Z,cz)
    r2=dx**2+dy**2+dz**2
    return amplitude*np.exp(-0.5*r2/sigma**2)

def normalize_field(u):
    u=np.asarray(u,float)
    u=u-u.min()
    return u/(u.max()+1e-30)

def make_initial_conditions_3d():
    rng=np.random.default_rng(42)
    z=rng.normal(0,1,(n,n,n))
    kk=2*np.pi*fftfreq(n,d=dx_um)
    kx,ky,kz=np.meshgrid(kk,kk,kk,indexing="ij")
    km=np.sqrt(kx**2+ky**2+kz**2)
    filt=np.exp(-(km/0.05)**4)
    random_smooth=normalize_field(np.real(ifftn(fftn(z)*filt)))

    one=normalize_field(gaussian3_periodic(L_um/2,L_um/2,L_um/2,220))

    many=np.zeros((n,n,n),float)
    centers=[
        (420,430,450),(850,700,1250),(1250,1250,1250),
        (1750,850,650),(2050,1850,1700),(700,1850,2000),
        (1750,1750,450),(450,1350,1750)
    ]
    for c0 in centers:
        many+=gaussian3_periodic(*c0,95)
    many=normalize_field(many)

    mixed=many.copy()
    singles=[
        (250,1150,350),(1100,300,1950),(1500,2100,1200),
        (2200,900,2200),(600,2200,850)
    ]
    for c0 in singles:
        mixed+=0.45*gaussian3_periodic(*c0,40)
    mixed=normalize_field(mixed)

    return {
        "Random smooth":random_smooth,
        "One central cluster":one,
        "Many clusters":many,
        "Clusters + singles":mixed
    }

ICs=make_initial_conditions_3d()

mid=n//2
fig,axs=plt.subplots(2,2,figsize=(10,9))
for ax,(name,u0) in zip(axs.ravel(),ICs.items()):
    im=ax.imshow(u0[:,:,mid].T,origin="lower",extent=[0,L_um,0,L_um],aspect="equal")
    ax.set_title(name)
    ax.set_xlabel("x (um)")
    ax.set_ylabel("y (um)")
    fig.colorbar(im,ax=ax,fraction=.046,pad=.04)
fig.suptitle("Central z-slices of 3D initial conditions",fontweight="bold")
plt.tight_layout()
plt.show()


## 4. 3D Fourier symbols and exact linear evolution

In [ ]:

k1=2*np.pi*fftfreq(n,d=dx_um)
KX,KY,KZ=np.meshgrid(k1,k1,k1,indexing="ij")
K2=KX**2+KY**2+KZ**2
KMAG=np.sqrt(K2)

def model_parameters(tag):
    return MODEL_LIBRARY[tag]["true"].copy()

def operator_symbol_3d(op,p):
    if op=="Diff": return -p["D"]*K2
    if op=="Adv": return -1j*p["c"]*KX
    if op=="Hyper": return -p["nu"]*K2**2
    if op=="SH": return -KAPPA_SH*(1-p["ell"]**2*K2)**2
    if op=="FracDiff": return -p["Df"]*KMAG**p["alpha"]
    if op=="Disp": return 1j*p["beta3"]*KX**3
    if op=="Sixth": return -p["mu6"]*K2**3
    if op=="Loss": return -p["eta"]+0*K2
    if op=="Nonlocal": return p["chi"]*(np.exp(-0.5*p["sigma"]**2*K2)-1)
    raise ValueError(op)

def complex_symbol_3d(tag,p):
    spec=MODEL_LIBRARY[tag]
    if spec["kind"]=="CH":
        return p["A2"]*K2-p["A4"]*K2**2

    lam=np.zeros_like(K2,dtype=complex)+p.get("R",0.0)

    if spec["kind"]=="KS":
        return lam+p["a2"]*K2-p["a4"]*K2**2

    for op in spec["ops"]:
        lam+=operator_symbol_3d(op,p)
    return lam

def real_symbol_radial(k,tag,p):
    k=np.asarray(k,float)
    spec=MODEL_LIBRARY[tag]

    if spec["kind"]=="CH":
        return p["A2"]*k**2-p["A4"]*k**4

    lam=np.zeros_like(k)+p.get("R",0.0)

    if spec["kind"]=="KS":
        return lam+p["a2"]*k**2-p["a4"]*k**4

    for op in spec["ops"]:
        if op=="Diff": lam-=p["D"]*k**2
        elif op=="Hyper": lam-=p["nu"]*k**4
        elif op=="SH": lam-=KAPPA_SH*(1-p["ell"]**2*k**2)**2
        elif op=="FracDiff": lam-=p["Df"]*np.abs(k)**p["alpha"]
        elif op=="Sixth": lam-=p["mu6"]*k**6
        elif op=="Loss": lam-=p["eta"]
        elif op=="Nonlocal": lam+=p["chi"]*(np.exp(-0.5*p["sigma"]**2*k**2)-1)
        # Adv and Disp are purely imaginary.
    return lam

def simulate_linear_3d(u0,tag,p):
    lam=complex_symbol_3d(tag,p)
    growth=np.exp(lam*dt)
    u=np.asarray(u0,float).copy()
    stack=[]
    for _ in range(n_frames):
        stack.append(u.copy())
        u=np.real(ifftn(fftn(u)*growth))
    return np.asarray(stack)


## 5. 3D radial PSD

In [ ]:

def radial_psd_3d(u,dx=dx_um):
    # Spherical-shell average of the 3D Fourier power.
    u=np.asarray(u,float)
    m=u.shape[0]
    z=u-np.mean(u)
    F=fftn(z)
    P=np.abs(F)**2/z.size

    kk=2*np.pi*fftfreq(m,d=dx)
    kx,ky,kz=np.meshgrid(kk,kk,kk,indexing="ij")
    km=np.sqrt(kx**2+ky**2+kz**2)

    dk=2*np.pi/(m*dx)
    bins=np.floor(km.ravel()/dk+0.5).astype(int)

    psum=np.bincount(bins,weights=P.ravel())
    count=np.bincount(bins)
    ksum=np.bincount(bins,weights=km.ravel())

    good=count>0
    kval=ksum[good]/count[good]
    pval=psum[good]/count[good]

    keep=kval>0
    return kval[keep],pval[keep]

def normalize_radial_psd(k,P):
    use=np.isfinite(k)&np.isfinite(P)&(P>0)&(k>0)
    if use.sum()<3:
        return np.full_like(P,np.nan,float),np.nan
    norm=np.trapezoid(P[use],k[use])
    return P/(norm+1e-30),norm



## 5A. Visualizing how the PDE changes the initial 3D condition

For each PDE, the initial volume \(u(\mathbf{x},0)\) is evolved using

\[
\widehat u(\mathbf{k},t+\Delta t)
=
\widehat u(\mathbf{k},t)e^{\lambda(\mathbf{k})\Delta t}.
\]

The plots below show the same evolution in three complementary ways:

1. **Absolute central \(z\)-slices** — preserve growth/decay amplitude.
2. **Normalized central \(z\)-slices** — emphasize morphology and spatial redistribution.
3. **Radial PSD curves** — show which spatial frequencies are amplified or suppressed.

This makes the effect of each PDE visible directly from the initial condition to later times.


In [ ]:

EVOLUTION_FRAMES=[0,5,10,15,20,29]

PLOT_EVOLUTION_FOR_ALL_MODELS=False

EVOLUTION_MODELS=[
    "R+Diff",
    "R+Adv",
    "R+Hyper",
    "R+SH",
    "R+FracDiff",
    "R+KSlin",
    "CHlin"
]

def normalize_snapshot(u):
    u=np.asarray(u,float)
    return (u-np.nanmin(u))/(np.nanmax(u)-np.nanmin(u)+1e-30)

def plot_pde_evolution_3d(
    u0,
    tag,
    ic_name,
    frames=EVOLUTION_FRAMES,
    show_absolute=True,
    show_normalized=True,
    show_psd=True
):
    p=model_parameters(tag)
    stack=simulate_linear_3d(u0,tag,p)
    mid=stack.shape[-1]//2

    if show_absolute:
        selected=[stack[f,:,:,mid] for f in frames]
        vmin=min(np.nanmin(a) for a in selected)
        vmax=max(np.nanmax(a) for a in selected)

        fig,axs=plt.subplots(
            1,len(frames),
            figsize=(3.0*len(frames),3.2),
            constrained_layout=True
        )
        axs=np.atleast_1d(axs)

        for ax,fid in zip(axs,frames):
            im=ax.imshow(
                stack[fid,:,:,mid].T,
                origin="lower",
                extent=[0,L_um,0,L_um],
                aspect="equal",
                vmin=vmin,
                vmax=vmax
            )
            ax.set_title(f"t={times[fid]:.2f}")
            ax.set_xlabel("x (um)")
        axs[0].set_ylabel("y (um)")

        fig.colorbar(im,ax=axs.tolist(),shrink=.78,label="u")
        fig.suptitle(
            f"{tag} | {ic_name}\nAbsolute PDE evolution",
            fontweight="bold"
        )
        plt.show()

    if show_normalized:
        fig,axs=plt.subplots(
            1,len(frames),
            figsize=(3.0*len(frames),3.2),
            constrained_layout=True
        )
        axs=np.atleast_1d(axs)

        for ax,fid in zip(axs,frames):
            us=normalize_snapshot(stack[fid])
            im=ax.imshow(
                us[:,:,mid].T,
                origin="lower",
                extent=[0,L_um,0,L_um],
                aspect="equal",
                vmin=0,
                vmax=1
            )
            ax.set_title(f"t={times[fid]:.2f}")
            ax.set_xlabel("x (um)")
        axs[0].set_ylabel("y (um)")

        fig.colorbar(im,ax=axs.tolist(),shrink=.78,label="Normalized field")
        fig.suptitle(
            f"{tag} | {ic_name}\nNormalized morphology",
            fontweight="bold"
        )
        plt.show()

    if show_psd:
        fig,ax=plt.subplots(figsize=(8,5))
        for fid in frames:
            k,P=radial_psd_3d(stack[fid])
            use=np.isfinite(k)&np.isfinite(P)&(P>0)&(k>0)&(k<=K_MAX_SNAPSHOT)
            if np.any(use):
                ax.semilogy(
                    k[use],P[use],
                    linewidth=1.6,
                    label=f"t={times[fid]:.2f}"
                )

        ax.set_xlabel("Radial wavenumber k")
        ax.set_ylabel("3D radial PSD")
        ax.set_title(f"{tag} | {ic_name} — PSD evolution")
        ax.grid(alpha=.25)
        ax.legend(fontsize=8,ncol=2)
        plt.tight_layout()
        plt.show()

    return stack

def plot_initial_vs_final_3d(u0,tag,ic_name,final_frame=29):
    p=model_parameters(tag)
    stack=simulate_linear_3d(u0,tag,p)
    mid=stack.shape[-1]//2

    arr0=stack[0,:,:,mid]
    arr1=stack[final_frame,:,:,mid]

    vmin=min(arr0.min(),arr1.min())
    vmax=max(arr0.max(),arr1.max())

    fig,axs=plt.subplots(1,2,figsize=(8,3.8),constrained_layout=True)

    im=axs[0].imshow(
        arr0.T,origin="lower",
        extent=[0,L_um,0,L_um],
        aspect="equal",
        vmin=vmin,vmax=vmax
    )
    axs[0].set_title("Initial")
    axs[0].set_xlabel("x (um)")
    axs[0].set_ylabel("y (um)")

    axs[1].imshow(
        arr1.T,origin="lower",
        extent=[0,L_um,0,L_um],
        aspect="equal",
        vmin=vmin,vmax=vmax
    )
    axs[1].set_title(f"After PDE, t={times[final_frame]:.2f}")
    axs[1].set_xlabel("x (um)")

    fig.colorbar(im,ax=axs.tolist(),shrink=.8,label="u")
    fig.suptitle(f"{tag} | {ic_name}: initial vs evolved",fontweight="bold")
    plt.show()

    return stack



### Run the evolution plots

By default, the notebook plots seven representative PDEs for all four initial conditions.  
Set

```python
PLOT_EVOLUTION_FOR_ALL_MODELS = True
```

to generate the same plots for all 53 PDEs.


In [ ]:

# Dependency check for the evolution plotting section
required_names = [
    "radial_psd_3d",
    "model_parameters",
    "simulate_linear_3d",
    "MODEL_LIBRARY",
    "MODEL_VARIANTS",
    "ICs"
]

missing=[name for name in required_names if name not in globals()]
if missing:
    raise RuntimeError(
        "Please run the notebook from the top before this section. "
        f"Missing definitions: {missing}"
    )

print("Evolution plotting dependencies are available.")


In [ ]:

models_to_plot=MODEL_VARIANTS if PLOT_EVOLUTION_FOR_ALL_MODELS else EVOLUTION_MODELS

for tag in models_to_plot:
    if tag not in MODEL_LIBRARY:
        continue

    for ic_name,u0 in ICs.items():
        print(f"Evolution: {tag} | {ic_name}")
        _=plot_pde_evolution_3d(
            u0,
            tag,
            ic_name,
            frames=EVOLUTION_FRAMES,
            show_absolute=True,
            show_normalized=True,
            show_psd=True
        )
        plt.close("all")


## 6. Identifiability and parameter fitting

In [ ]:

def invisible_names(tag):
    out=[]
    for op in MODEL_LIBRARY[tag]["ops"]:
        if op=="Adv": out.append("c")
        if op=="Disp": out.append("beta3")
    return list(dict.fromkeys(out))

def primitive_confounded(tag):
    ops=set(MODEL_LIBRARY[tag]["ops"])
    return "SH" in ops and ("Diff" in ops or "Hyper" in ops)

def effective_poly_truth(tag):
    p=MODEL_LIBRARY[tag]["true"]; ops=set(MODEL_LIBRARY[tag]["ops"])
    c0=p.get("R",0.0); c2=c4=c6=0.0
    if "Loss" in ops: c0-=p["eta"]
    if "Diff" in ops: c2-=p["D"]
    if "Hyper" in ops: c4-=p["nu"]
    if "Sixth" in ops: c6-=p["mu6"]
    if "SH" in ops:
        c0-=KAPPA_SH
        c2+=2*KAPPA_SH*p["ell"]**2
        c4-=KAPPA_SH*p["ell"]**4
    return {"c0":c0,"c2":c2,"c4":c4,"c6":c6}

def identifiable_names(tag):
    spec=MODEL_LIBRARY[tag]; ops=spec["ops"]
    if spec["kind"]=="CH": return ["A2","A4"]
    if spec["kind"]=="KS": return ["R","a2","a4"]
    if primitive_confounded(tag):
        names=["c0","c2","c4"]
        if "Sixth" in ops: names.append("c6")
        return names
    names=["R"]
    for op in ops:
        if op=="Diff": names.append("D")
        elif op=="Hyper": names.append("nu")
        elif op=="SH": names.append("ell")
        elif op=="FracDiff": names+=["Df","alpha"]
        elif op=="Sixth": names.append("mu6")
        elif op=="Loss":
            if "R" in names: names.remove("R")
            names.append("R_eff")
        elif op=="Nonlocal": names+=["chi","sigma"]
    return list(dict.fromkeys(names))

def true_identifiable_value(tag,name):
    p=MODEL_LIBRARY[tag]["true"]
    if name in p: return p[name]
    if name=="R_eff": return p["R"]-p["eta"]
    if name in ("c0","c2","c4","c6"): return effective_poly_truth(tag)[name]
    return np.nan

def relative_error(est,true):
    return 100*(est-true)/(abs(true)+1e-30)

def fit_real_symbol(k,y,tag):
    k=np.asarray(k,float); y=np.asarray(y,float)
    names=identifiable_names(tag); true=MODEL_LIBRARY[tag]["true"]

    if len(names)==1 and names[0] in ("R","R_eff"):
        val=float(np.mean(y))
        return {names[0]:val},np.nan,np.full_like(y,val)

    if any(nm in names for nm in ("c0","c2","c4","c6")):
        degree=3 if "c6" in names else 2
        cf=np.polyfit(k**2,y,degree)
        pred=np.polyval(cf,k**2)
        asc=cf[::-1]
        fit={"c0":asc[0],"c2":asc[1],"c4":asc[2]}
        if degree==3: fit["c6"]=asc[3]
        ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
        return fit,(1-ssr/sst if sst>0 else np.nan),pred

    def unpack(theta):
        p=true.copy()
        for nm,val in zip(names,theta):
            if nm=="R_eff": p["R"]=float(val)+p["eta"]
            else: p[nm]=float(val)
        return p

    p0=[]; lo=[]; hi=[]
    for nm in names:
        tv=true_identifiable_value(tag,nm)
        if nm in ("R","R_eff","a2","A2"):
            p0.append(tv*.9 if tv!=0 else .01); lo.append(-20*abs(tv)-10); hi.append(20*abs(tv)+10)
        elif nm=="alpha":
            p0.append(tv*.9); lo.append(.2); hi.append(6)
        elif nm in ("ell","sigma"):
            p0.append(tv*.9); lo.append(.1); hi.append(500)
        else:
            p0.append(tv*.9); lo.append(0); hi.append(max(1,20*abs(tv)))

    def f(k,*theta):
        return real_symbol_radial(k,tag,unpack(theta))

    popt,_=curve_fit(f,k,y,p0=p0,bounds=(lo,hi),maxfev=300000)
    pred=f(k,*popt)
    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    R2=1-ssr/sst if sst>0 else np.nan
    return dict(zip(names,popt)),R2,pred


## 7. Scenario 3: unknown elapsed time

In [ ]:

def recover_S3(k,y,tag):
    k=np.asarray(k,float); y=np.asarray(y,float)
    spec=MODEL_LIBRARY[tag]
    ops=tuple(o for o in spec["ops"] if o not in ("Adv","Disp"))

    if spec["kind"]=="CH":
        cf=np.polyfit(k**2,y,2); pred=np.polyval(cf,k**2)
        b4,b2,b0=cf
        return ({"A2/A4":-b2/b4} if abs(b4)>1e-30 else {}),pred

    if spec["kind"]=="KS":
        cf=np.polyfit(k**2,y,2); pred=np.polyval(cf,k**2)
        b4,b2,b0=cf; rec={}
        if abs(b0)>1e-30:
            rec["a2/R"]=b2/b0
            rec["a4/R"]=-b4/b0
        return rec,pred

    if ops==("SH",):
        cf=np.polyfit(k**2,y,2); pred=np.polyval(cf,k**2)
        c0,c2,c4=cf[::-1]; rec={}
        if abs(c2)>1e-30:
            ell2=-2*c4/c2
            if ell2>0:
                ell=np.sqrt(ell2)
                s=c2/(2*KAPPA_SH*ell2)
                if abs(s)>1e-30:
                    rec={"ell":ell,"2*dt":s,"R":c0/s+KAPPA_SH}
        return rec,pred

    if "SH" in ops:
        degree=3 if "Sixth" in ops else 2
        cf=np.polyfit(k**2,y,degree); pred=np.polyval(cf,k**2)
        asc=cf[::-1]; c0=asc[0]
        rec={"note":"effective polynomial ratios only"}
        if abs(c0)>1e-30:
            if len(asc)>1: rec["net_k2/c0"]=asc[1]/c0
            if len(asc)>2: rec["net_k4/c0"]=asc[2]/c0
            if len(asc)>3: rec["net_k6/c0"]=asc[3]/c0
        return rec,pred

    has_frac="FracDiff" in ops
    has_nonlocal="Nonlocal" in ops
    has_diff="Diff" in ops
    has_hyper="Hyper" in ops
    has_sixth="Sixth" in ops
    has_loss="Loss" in ops

    if has_frac or has_nonlocal:
        def model(k,*th):
            it=iter(th)
            A=next(it)
            out=np.full_like(k,A,float)
            if has_diff: out+=next(it)*k**2
            if has_hyper: out+=next(it)*k**4
            if has_sixth: out+=next(it)*k**6
            if has_frac:
                B=next(it); alpha=next(it)
                out+=B*np.abs(k)**alpha
            if has_nonlocal:
                C=next(it); sigma=next(it)
                out+=C*(np.exp(-.5*sigma**2*k**2)-1)
            return out

        p0=[np.mean(y)]; lo=[-np.inf]; hi=[np.inf]
        if has_diff: p0+= [-100]; lo+= [-np.inf]; hi+=[np.inf]
        if has_hyper: p0+= [-1e5]; lo+= [-np.inf]; hi+=[np.inf]
        if has_sixth: p0+= [-1e7]; lo+= [-np.inf]; hi+=[np.inf]
        if has_frac: p0+= [-100,1.5]; lo+= [-np.inf,.2]; hi+=[np.inf,6]
        if has_nonlocal: p0+= [-1,45]; lo+= [-np.inf,.1]; hi+=[np.inf,500]

        try:
            popt,_=curve_fit(model,k,y,p0=p0,bounds=(lo,hi),maxfev=300000)
            pred=model(k,*popt)
        except Exception:
            return {"note":"non-polynomial fit failed"},np.full_like(y,np.mean(y))

        it=iter(popt); A=next(it); rec={}
        if has_diff:
            B=next(it)
            if abs(A)>1e-30: rec["D/c0"]=-B/A
        if has_hyper:
            B=next(it)
            if abs(A)>1e-30: rec["nu/c0"]=-B/A
        if has_sixth:
            B=next(it)
            if abs(A)>1e-30: rec["mu6/c0"]=-B/A
        if has_frac:
            B=next(it); alpha=next(it)
            rec["alpha"]=alpha
            if abs(A)>1e-30: rec["Df/c0"]=-B/A
        if has_nonlocal:
            C=next(it); sigma=next(it)
            rec["sigma"]=sigma
            if abs(A)>1e-30: rec["chi/c0"]=C/A
        return rec,pred

    degree=0
    if has_diff: degree=max(degree,1)
    if has_hyper: degree=max(degree,2)
    if has_sixth: degree=max(degree,3)

    if degree==0:
        return {"note":"constant-only real symbol; unknown-time scale unidentifiable"},np.full_like(y,np.mean(y))

    cf=np.polyfit(k**2,y,degree)
    pred=np.polyval(cf,k**2)
    asc=cf[::-1]
    c0=asc[0]
    label="R-eta" if has_loss else "R"
    rec={}
    if abs(c0)>1e-30:
        if has_diff: rec[f"D/({label})"]=-asc[1]/c0
        if has_hyper: rec[f"nu/({label})"]=-asc[2]/c0
        if has_sixth: rec[f"mu6/({label})"]=-asc[3]/c0
    return rec,pred


## 8. Scenarios 1–5

In [ ]:

def scenario1(stack,tag,kmax=K_MAX):
    kref,_=radial_psd_3d(stack[0])
    P=np.asarray([radial_psd_3d(fr)[1] for fr in stack])

    lam=np.full(len(kref),np.nan)
    temporal_r2=np.full(len(kref),np.nan)

    for j in range(len(kref)):
        valid=np.isfinite(P[:,j])&(P[:,j]>0)
        if valid.sum()<6: continue
        lr=linregress(times[valid],np.log(P[valid,j]))
        lam[j]=lr.slope/2
        temporal_r2[j]=lr.rvalue**2

    use=np.isfinite(lam)&(kref>0)&(kref<kmax)&(temporal_r2>0.5)
    if use.sum()<5: return None

    fit,R2,pred=fit_real_symbol(kref[use],lam[use],tag)
    return {"k":kref[use],"lambda":lam[use],"pred":pred,"fit":fit,"R2":R2,
            "temporal_R2":temporal_r2[use]}

def scenario2(stack,tag,frame_t=FRAME_T,kmax=K_MAX):
    k,P0=radial_psd_3d(stack[0])
    _,Pt=radial_psd_3d(stack[frame_t])
    elapsed=times[frame_t]
    use=np.isfinite(P0)&np.isfinite(Pt)&(P0>0)&(Pt>0)&(k>0)&(k<kmax)
    kv=k[use]
    lam=np.log(Pt[use]/P0[use])/(2*elapsed)
    fit,R2,pred=fit_real_symbol(kv,lam,tag)
    return {"k":kv,"lambda":lam,"pred":pred,"fit":fit,"R2":R2}

def scenario3(stack,tag,frame_t=FRAME_T,kmax=K_MAX):
    k,P0=radial_psd_3d(stack[0])
    _,Pt=radial_psd_3d(stack[frame_t])
    use=np.isfinite(P0)&np.isfinite(Pt)&(P0>0)&(Pt>0)&(k>0)&(k<kmax)
    kv=k[use]
    y=np.log(Pt[use]/P0[use])
    rec,pred=recover_S3(kv,y,tag)
    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    R2=1-ssr/sst if sst>0 else np.nan
    return {"k":kv,"y":y,"pred":pred,"recovered":rec,"R2":R2}

def snapshot_degree(tag):
    spec=MODEL_LIBRARY[tag]; ops=spec["ops"]
    if spec["kind"] in ("KS","CH"): return 2
    if "Sixth" in ops: return 3
    if "Hyper" in ops or "SH" in ops: return 2
    if "Diff" in ops: return 1
    if "FracDiff" in ops or "Nonlocal" in ops: return 3
    return 0

def scenario4(stack,tag,frame=SNAP_FRAME,kmin=K_MIN_SNAPSHOT,kmax=K_MAX_SNAPSHOT):
    k,P=radial_psd_3d(stack[frame])
    Pn,_=normalize_radial_psd(k,P)
    use=np.isfinite(Pn)&(Pn>0)&(k>=kmin)&(k<=kmax)
    kv=k[use]; y=np.log(Pn[use]+1e-30)
    if len(kv)<5: return None

    degree=snapshot_degree(tag)
    if degree==0:
        coef=np.array([np.mean(y)])
        pred=np.full_like(y,coef[0])
    else:
        coef=np.polyfit(kv**2,y,degree)
        pred=np.polyval(coef,kv**2)

    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    R2=1-ssr/sst if sst>0 else np.nan
    return {"k":kv,"logP":y,"coef":coef,"pred":pred,"R2":R2,
            "status":"single-snapshot radial spectral-shape diagnostic only"}

def fit_cube_shape(cube,tag):
    if cube.size<64 or np.std(cube)<1e-10: return None
    k,P=radial_psd_3d(cube,dx=dx_um)
    Pn,_=normalize_radial_psd(k,P)
    use=np.isfinite(Pn)&(Pn>0)&(k>=K_MIN_SNAPSHOT)&(k<=K_MAX_SNAPSHOT)
    kv=k[use]; y=np.log(Pn[use]+1e-30)
    if len(kv)<4: return None

    degree=snapshot_degree(tag)
    if degree==0:
        coef=np.array([np.mean(y)])
        pred=np.full_like(y,coef[0])
    else:
        degree=min(degree,max(1,len(kv)-2))
        coef=np.polyfit(kv**2,y,degree)
        pred=np.polyval(coef,kv**2)

    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    R2=1-ssr/sst if sst>0 else np.nan
    metric=coef[-2] if len(coef)>=2 else coef[-1]
    return {"metric":float(metric),"R2":float(R2)}

def scenario5(stack,tag,frame=SNAP_FRAME,grid=GRID_N,r2_min=R2_MIN):
    vol=stack[frame]
    m=vol.shape[0]//grid
    rows=[]

    for i in range(grid):
        for j in range(grid):
            for q in range(grid):
                cube=vol[i*m:(i+1)*m,j*m:(j+1)*m,q*m:(q+1)*m]
                rr=fit_cube_shape(cube,tag)
                if rr is not None:
                    rows.append({"i":i,"j":j,"k":q,**rr})

    if not rows: return None

    df=pd.DataFrame(rows)
    good=df[np.isfinite(df["R2"])&(df["R2"]>=r2_min)]

    return {
        "df_tiles":df,
        "median_metric":float(np.nanmedian(good["metric"])) if len(good) else np.nan,
        "mean_metric":float(np.nanmean(good["metric"])) if len(good) else np.nan,
        "median_R2":float(np.nanmedian(good["R2"])) if len(good) else np.nan,
        "mean_R2":float(np.nanmean(good["R2"])) if len(good) else np.nan,
        "n_good":int(len(good)),
        "n_total":int(len(df)),
        "grid":grid
    }


## 9. Run all 53 PDEs across four 3D initial conditions

In [ ]:

ALL_RESULTS={}

for model_index,tag in enumerate(MODEL_VARIANTS,1):
    p=model_parameters(tag)
    print(f"[{model_index:02d}/{len(MODEL_VARIANTS)}] {tag}")

    res={"tag":tag,"true":p,"S1":{},"S2":{},"S3":{},"S4":{},"S5":{}}

    for ic,u0 in ICs.items():
        try:
            stack=simulate_linear_3d(u0,tag,p)
        except Exception as e:
            print(" simulation failed:",ic,e)
            for s in ("S1","S2","S3","S4","S5"): res[s][ic]=None
            continue

        for s,func in [("S1",scenario1),("S2",scenario2),("S3",scenario3),
                       ("S4",scenario4),("S5",scenario5)]:
            try:
                res[s][ic]=func(stack,tag)
            except Exception as e:
                res[s][ic]=None
                print(f"  {ic} {s} failed: {type(e).__name__}: {e}")

        if RUN_PER_MODEL_FIGURES:
            mid=n//2
            fig,axs=plt.subplots(1,len(FRAME_LIST),figsize=(16,3))
            for ax,fid in zip(axs,FRAME_LIST):
                im=ax.imshow(stack[fid,:,:,mid].T,origin="lower",aspect="equal")
                ax.set_title(f"t={times[fid]:.2f}")
                ax.set_xticks([]); ax.set_yticks([])
            fig.suptitle(f"{tag} — {ic} — central z-slice")
            plt.tight_layout(); plt.show()

        del stack

    ALL_RESULTS[tag]=res

print("Finished",len(ALL_RESULTS),"PDEs.")


## 10. Detailed 3D Scenario-5 reaction–diffusion grid sweep

In [ ]:

def fit_cube_RD(cube,R_known,elapsed_time):
    if cube.size<64 or np.std(cube)<1e-10: return None
    k,P=radial_psd_3d(cube,dx=dx_um)
    use=np.isfinite(P)&(P>0)&(k>K_MIN_SNAPSHOT)&(k<K_MAX_SNAPSHOT)
    kf=k[use]; Pf=P[use]
    if len(kf)<4: return None

    x2=kf**2
    y=np.log10(Pf+1e-30)
    Xmat=np.column_stack([np.ones_like(x2),x2])
    beta,*_=np.linalg.lstsq(Xmat,y,rcond=None)
    pred=Xmat@beta
    A,B=beta

    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    R2=1-ssr/sst if sst>0 else np.nan

    D_hat=-B*np.log(10)/(2*elapsed_time)
    log10_DR=np.log10(D_hat/R_known) if D_hat>0 and R_known>0 else np.nan
    return {"D_hat":float(D_hat),"log10_DR":float(log10_DR) if np.isfinite(log10_DR) else np.nan,
            "R2":float(R2)}

def fit_tiled_RD_3d(vol,R_known,elapsed_time,grid_n,r2_min=R2_MIN):
    m=vol.shape[0]//grid_n
    rows=[]
    for i in range(grid_n):
        for j in range(grid_n):
            for q in range(grid_n):
                cube=vol[i*m:(i+1)*m,j*m:(j+1)*m,q*m:(q+1)*m]
                rr=fit_cube_RD(cube,R_known,elapsed_time)
                if rr is not None:
                    rows.append({"i":i,"j":j,"k":q,**rr})

    if not rows: return None
    df=pd.DataFrame(rows)
    good=df[np.isfinite(df["log10_DR"])&np.isfinite(df["R2"])&(df["R2"]>=r2_min)]
    return {
        "median":float(np.median(good["log10_DR"])) if len(good) else np.nan,
        "mean":float(np.mean(good["log10_DR"])) if len(good) else np.nan,
        "std":float(np.std(good["log10_DR"])) if len(good) else np.nan,
        "n_good":int(len(good)),
        "n_total":int(len(df)),
        "df_tiles":df
    }

S5_RD_TAG="R+Diff"
R_TRUE=MODEL_LIBRARY[S5_RD_TAG]["true"]["R"]
D_TRUE=MODEL_LIBRARY[S5_RD_TAG]["true"]["D"]
TRUE_LOG10_DR=np.log10(D_TRUE/R_TRUE)
ELAPSED_S5=times[SNAP_FRAME]

rows=[]
for ic,u0 in ICs.items():
    stack=simulate_linear_3d(u0,S5_RD_TAG,MODEL_LIBRARY[S5_RD_TAG]["true"])
    vol=stack[SNAP_FRAME]
    for grid_n in GRID_LIST:
        rr=fit_tiled_RD_3d(vol,R_TRUE,ELAPSED_S5,grid_n)
        rows.append({
            "Initial condition":ic,
            "Grid":grid_n,
            "Cubes":grid_n**3,
            "Median log10(D/R)":rr["median"] if rr else np.nan,
            "True log10(D/R)":TRUE_LOG10_DR,
            "Error":rr["median"]-TRUE_LOG10_DR if rr and np.isfinite(rr["median"]) else np.nan,
            "Good cubes":rr["n_good"] if rr else 0,
            "Total fitted cubes":rr["n_total"] if rr else 0
        })
    del stack

S5_RD_GRID_DF=pd.DataFrame(rows)
display(S5_RD_GRID_DF.round(4))

fig,ax=plt.subplots(figsize=(9,5))
for ic,grp in S5_RD_GRID_DF.groupby("Initial condition"):
    grp=grp.sort_values("Grid")
    ax.plot(grp["Grid"],grp["Median log10(D/R)"],marker="o",label=ic)
ax.axhline(TRUE_LOG10_DR,linestyle="--",label="True log10(D/R)")
ax.set_xlabel("Cubes per axis")
ax.set_ylabel("Median log10(D/R)")
ax.set_xticks(GRID_LIST)
ax.grid(alpha=.25)
ax.legend(fontsize=8)
ax.set_title("3D Scenario 5 reaction-diffusion tiled estimator")
plt.tight_layout()
plt.show()


## 11. Master summary: 53 PDEs × 5 scenarios × 4 initial conditions

In [ ]:

def fmt_dict(d):
    if not isinstance(d,dict): return ""
    out=[]
    for k,v in d.items():
        if isinstance(v,str):
            out.append(f"{k}: {v}")
        else:
            try: out.append(f"{k}={float(v):.5g}")
            except Exception: out.append(f"{k}={v}")
    return "; ".join(out)

MASTER_ROWS=[]

for tag,res in ALL_RESULTS.items():
    for scen in ("S1","S2","S3","S4","S5"):
        for ic in ICs:
            rr=res[scen].get(ic)
            row={
                "PDE":tag,
                "Operators":" + ".join(MODEL_LIBRARY[tag]["ops"]),
                "Scenario":scen,
                "Initial condition":ic,
                "True parameters":fmt_dict(res["true"]),
                "Recovered / metric":"",
                "R2":np.nan,
                "Median abs parameter error (%)":np.nan,
                "Good cubes":np.nan,
                "Total cubes":np.nan,
                "PSD-invisible":"; ".join(invisible_names(tag)),
                "Status":""
            }

            if rr is None:
                row["Status"]="failed / unavailable"
            elif scen in ("S1","S2"):
                row["Recovered / metric"]=fmt_dict(rr["fit"])
                row["R2"]=rr["R2"]
                errs=[]
                for nm,est in rr["fit"].items():
                    tv=true_identifiable_value(tag,nm)
                    if np.isfinite(tv) and abs(tv)>1e-30:
                        errs.append(abs(relative_error(est,tv)))
                row["Median abs parameter error (%)"]=np.nanmedian(errs) if errs else np.nan
                row["Status"]="known-time radial-PSD symbol recovery"
            elif scen=="S3":
                row["Recovered / metric"]=fmt_dict(rr["recovered"])
                row["R2"]=rr["R2"]
                row["Status"]="unknown-time ratio / shape recovery"
            elif scen=="S4":
                row["Recovered / metric"]="shape coef="+np.array2string(np.asarray(rr["coef"]),precision=4)
                row["R2"]=rr["R2"]
                row["Status"]="single-volume shape diagnostic"
            elif scen=="S5":
                row["Recovered / metric"]=f"median accepted-cube metric={rr['median_metric']:.5g}"
                row["R2"]=rr["median_R2"]
                row["Good cubes"]=rr["n_good"]
                row["Total cubes"]=rr["n_total"]
                row["Status"]="3D tiled regional shape diagnostic"

            MASTER_ROWS.append(row)

MASTER_SUMMARY_DF=pd.DataFrame(MASTER_ROWS)
assert len(MASTER_SUMMARY_DF)==53*5*4

MASTER_R2_MATRIX=MASTER_SUMMARY_DF.pivot_table(
    index=["PDE","Initial condition"],columns="Scenario",values="R2",aggfunc="first"
).reindex(columns=["S1","S2","S3","S4","S5"])

PDE_SCENARIO_MEDIAN_R2=MASTER_SUMMARY_DF.groupby(
    ["PDE","Scenario"],as_index=False
)["R2"].median().pivot(index="PDE",columns="Scenario",values="R2").reindex(
    columns=["S1","S2","S3","S4","S5"]
)

print("Master rows:",len(MASTER_SUMMARY_DF))
display(MASTER_SUMMARY_DF)
display(MASTER_R2_MATRIX.round(4))
display(PDE_SCENARIO_MEDIAN_R2.round(4))


# 12. Five-scenario plots for every PDE

In [ ]:

FIVE_SCENARIO_R2_TABLES={}

for tag,res in ALL_RESULTS.items():
    ic_names=list(ICs)
    scen_names=["S1","S2","S3","S4","S5"]
    table=np.full((4,5),np.nan)

    for i,ic in enumerate(ic_names):
        for j,scen in enumerate(("S1","S2","S3","S4")):
            rr=res[scen].get(ic)
            if rr is not None:
                table[i,j]=rr.get("R2",np.nan)
        rr5=res["S5"].get(ic)
        if rr5 is not None:
            table[i,4]=rr5.get("median_R2",np.nan)

    df=pd.DataFrame(table,index=ic_names,columns=scen_names)
    FIVE_SCENARIO_R2_TABLES[tag]=df

    fig,ax=plt.subplots(figsize=(10,5))
    xloc=np.arange(5)
    width=.18
    offsets=np.linspace(-1.5*width,1.5*width,4)

    for off,ic in zip(offsets,ic_names):
        ax.bar(xloc+off,df.loc[ic].values,width,label=ic)

    ax.set_xticks(xloc)
    ax.set_xticklabels(["S1\nFull series","S2\nKnown time","S3\nUnknown time",
                        "S4\n1 volume","S5\n3D tiled"])
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Fit quality R2")
    ax.set_title(f"{tag} — five-scenario comparison")
    ax.grid(axis="y",alpha=.25)
    ax.legend(fontsize=8,ncol=2)
    plt.tight_layout()
    plt.show()


# 13. Final comparison plots across all PDEs

In [ ]:

# 13A. Theoretical radial dispersion signatures
n_models=len(MODEL_LIBRARY)
ncols=4
nrows=int(np.ceil(n_models/ncols))
fig,axs=plt.subplots(nrows,ncols,figsize=(18,3.0*nrows))
axs=np.atleast_1d(axs).ravel()
kplot=np.linspace(0,0.08,400)

for ax,(tag,spec) in zip(axs,MODEL_LIBRARY.items()):
    ax.plot(kplot,real_symbol_radial(kplot,tag,spec["true"]),linewidth=1.7)
    ax.axhline(0,linewidth=.8)
    ax.set_title(tag,fontsize=8)
    ax.set_xlabel("radial k")
    ax.set_ylabel("Re lambda(k)")
    ax.grid(alpha=.2)

for ax in axs[n_models:]:
    ax.axis("off")

fig.suptitle("Theoretical PSD-visible radial dispersion signatures — 3D",fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

# 13B. S1/S2 known-time parameter recovery error
PARAM_ERROR_DF=(
    MASTER_SUMMARY_DF[MASTER_SUMMARY_DF["Scenario"].isin(["S1","S2"])]
    .groupby(["PDE","Scenario"],as_index=False)["Median abs parameter error (%)"]
    .median()
)

pivot=PARAM_ERROR_DF.pivot(
    index="PDE",columns="Scenario",values="Median abs parameter error (%)"
).reindex(columns=["S1","S2"])

fig,ax=plt.subplots(figsize=(9,max(10,.30*len(pivot))))
im=ax.imshow(pivot.values,aspect="auto")
ax.set_xticks(range(2)); ax.set_xticklabels(["S1","S2"])
ax.set_yticks(range(len(pivot))); ax.set_yticklabels(pivot.index,fontsize=7)

for i in range(pivot.shape[0]):
    for j in range(2):
        v=pivot.iloc[i,j]
        if np.isfinite(v):
            ax.text(j,i,f"{v:.1f}",ha="center",va="center",fontsize=5.5)

fig.colorbar(im,ax=ax,label="Median absolute parameter error (%)")
ax.set_title("3D known-time parameter recovery error")
plt.tight_layout()
plt.show()


In [ ]:

# 13C. Median R2 across the four ICs
R2_MEDIAN_FULL=PDE_SCENARIO_MEDIAN_R2

fig,ax=plt.subplots(figsize=(11,max(10,.30*len(R2_MEDIAN_FULL))))
im=ax.imshow(R2_MEDIAN_FULL.values,aspect="auto",vmin=0,vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels(["S1","S2","S3","S4","S5"])
ax.set_yticks(range(len(R2_MEDIAN_FULL)))
ax.set_yticklabels(R2_MEDIAN_FULL.index,fontsize=7)

for i in range(R2_MEDIAN_FULL.shape[0]):
    for j in range(5):
        v=R2_MEDIAN_FULL.iloc[i,j]
        if np.isfinite(v):
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=5)

fig.colorbar(im,ax=ax,label="Median R2")
ax.set_title("Scenario-wise fit quality across all 3D PDE families")
plt.tight_layout()
plt.show()


In [ ]:

# 13D. One heatmap per scenario
for scen in ["S1","S2","S3","S4","S5"]:
    pivot=MASTER_SUMMARY_DF[
        MASTER_SUMMARY_DF["Scenario"]==scen
    ].pivot(index="PDE",columns="Initial condition",values="R2").reindex(columns=list(ICs))

    fig,ax=plt.subplots(figsize=(10,max(10,.30*len(pivot))))
    im=ax.imshow(pivot.values,aspect="auto",vmin=0,vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns,rotation=30,ha="right",fontsize=8)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index,fontsize=7)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v=pivot.iloc[i,j]
            if np.isfinite(v):
                ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=5)

    fig.colorbar(im,ax=ax,label="R2")
    ax.set_title(f"{scen} — 3D fit quality by PDE and initial condition")
    plt.tight_layout()
    plt.show()


In [ ]:

# 13E. Scenario 5 accepted-cube fractions
rows=[]
for tag,res in ALL_RESULTS.items():
    for ic,rr in res["S5"].items():
        frac=rr["n_good"]/rr["n_total"] if rr is not None and rr["n_total"] else np.nan
        rows.append({"PDE":tag,"Initial condition":ic,"Accepted fraction":frac})

S5_ACCEPT_DF=pd.DataFrame(rows)
pivot=S5_ACCEPT_DF.pivot(
    index="PDE",columns="Initial condition",values="Accepted fraction"
).reindex(columns=list(ICs))

fig,ax=plt.subplots(figsize=(10,max(10,.30*len(pivot))))
im=ax.imshow(pivot.values,aspect="auto",vmin=0,vmax=1)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns,rotation=30,ha="right",fontsize=8)
ax.set_yticks(range(len(pivot)))
ax.set_yticklabels(pivot.index,fontsize=7)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v=pivot.iloc[i,j]
        if np.isfinite(v):
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=5)

fig.colorbar(im,ax=ax,label="Accepted cube fraction")
ax.set_title(f"3D Scenario 5 accepted cubes (R2 >= {R2_MIN:.2f})")
plt.tight_layout()
plt.show()


In [ ]:

# 13F. Scenario-specific ranking tables
SCENARIO_RANKINGS={}

for scen in ["S1","S2","S3","S4","S5"]:
    rank=(
        MASTER_SUMMARY_DF[MASTER_SUMMARY_DF["Scenario"]==scen]
        .groupby("PDE",as_index=False)["R2"]
        .median()
        .rename(columns={"R2":"Median_R2"})
        .sort_values("Median_R2",ascending=False)
        .reset_index(drop=True)
    )
    rank["Rank"]=np.arange(1,len(rank)+1)
    SCENARIO_RANKINGS[scen]=rank[["Rank","PDE","Median_R2"]]
    print(f"\n{scen} top 20:")
    display(SCENARIO_RANKINGS[scen].head(20))



# 14. Interpretation and 3D cautions

- **S1:** full time-series radial PSD recovery.
- **S2:** two-volume radial PSD ratio with known elapsed time.
- **S3:** unknown-time ratios / shape information.
- **S4:** one-volume radial spectral-shape diagnostic only.
- **S5:** local cubic-tile spectral diagnostic. Only accepted cubes are used in the S5 summary.
- Radial averaging is natural for **isotropic** operators, but it discards directional information.
- \(x\)-directed advection and third-order dispersion remain phase-only and are therefore not identifiable from PSD magnitude.
- If experimental 3D data are anisotropic, use \(P(k_x,k_y,k_z)\), directional spectra, or angular sectors instead of relying only on spherical-shell averaging.
- FFT evolution assumes periodic boundaries. Experimental microscopy volumes may require detrending, tapering/windowing, or interior ROI selection before direct Fourier interpretation.
- Scenario 5 remains a robustness diagnostic; local tiling does not prove that unknown initial-spectrum bias vanishes.


# 15. Export all summary tables

In [ ]:

MASTER_SUMMARY_DF.to_csv("MASTER_3D_ALL_PDEs_ALL_5_SCENARIOS_tidy.csv",index=False)
MASTER_R2_MATRIX.to_csv("MASTER_3D_ALL_PDEs_R2_matrix.csv")
PDE_SCENARIO_MEDIAN_R2.to_csv("MASTER_3D_PDE_scenario_median_R2.csv")
PARAM_ERROR_DF.to_csv("MASTER_3D_known_time_parameter_error.csv",index=False)
S5_ACCEPT_DF.to_csv("MASTER_3D_S5_accepted_cube_fraction.csv",index=False)
S5_RD_GRID_DF.to_csv("MASTER_3D_S5_RD_grid_sweep.csv",index=False)

for scen,df in SCENARIO_RANKINGS.items():
    df.to_csv(f"MASTER_3D_{scen}_ranking.csv",index=False)

print("Saved all 3D summary CSV files.")
